In [ ]:
from pathlib import Path
import pandas as pd
import logging
from src.signals_scaling import (compute_moment_scaling_acc,compute_moment_scaling_vel, compute_moment_scaling_disp,
                         compute_scaling_exponents,test_scaling_linearity, fit_piecewise_scaling,
                         trim_to_event_window, compute_increment_tail_exponents, check_increments_distribution)
from src.plot_settings import set_plot_style
from src.plots import (plot_onset_diagnostic, plot_onset_distribution,
                       plot_increments_distribution)
colors = set_plot_style()
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
logger = logging.getLogger()
def check(condition, message):
    if condition:
        logger.info(message)
    else:
        raise ValueError(message)
logger.info("Environment ready")

In [ ]:
# ============================================================================
# Creazione di df_acc_long COMPLETAMENTE RAW (nessuna preprocessing)
# ============================================================================

import pandas as pd
from pathlib import Path

# 1. Carica i dati raw
print("Loading raw data...")
from src.io import build_accelerations
df_acc_raw = build_accelerations('../data/raw/query.zip')
print(f"Raw data loaded: {df_acc_raw['file'].nunique()} files, {len(df_acc_raw):,} samples")

# 2. Filtra solo i segnali lunghi (>= 48000 samples) - NESSUN'ALTRA MODIFICA
print("\nFiltering long signals...")
min_samples = 48000
signal_lengths = df_acc_raw.groupby('file')['sample'].max() + 1
long_files = signal_lengths[signal_lengths >= min_samples].index
df_acc_long = df_acc_raw[df_acc_raw['file'].isin(long_files)].copy()

print(f"Long signals: {df_acc_long['file'].nunique()} files")
print(f"Excluded: {df_acc_raw['file'].nunique() - df_acc_long['file'].nunique()} short files")

# 3. Verifica - dati completamente RAW
print("\n" + "="*70)
print("VERIFICATION - COMPLETELY RAW DATA:")
print("="*70)
mean_check = df_acc_long.groupby('file')['acceleration'].mean()
std_check = df_acc_long.groupby('file')['acceleration'].std()

print(f"\nMean per file (can be any value, NOT zero):")
print(f"  - Min mean:  {mean_check.min():.4f}")
print(f"  - Max mean:  {mean_check.max():.4f}")
print(f"  - Mean of means: {mean_check.mean():.4f}")

print(f"\nStd per file (should be >> 1 for raw data):")
print(f"  - Min std:  {std_check.min():.2f}")
print(f"  - Max std:  {std_check.max():.2f}")
print(f"  - Mean std: {std_check.mean():.2f}")

print(f"\nAcceleration range:")
print(f"  - Min: {df_acc_long['acceleration'].min():.2f} cm/s²")
print(f"  - Max: {df_acc_long['acceleration'].max():.2f} cm/s²")

print("\n✅ df_acc_long ready: COMPLETELY RAW (no baseline correction, no normalization)")
print(f"   {len(df_acc_long):,} samples, {df_acc_long['file'].nunique()} files")
print(f"   Columns: {df_acc_long.columns.tolist()}")
print("="*70)

In [ ]:
######### VERSIONE SENZA NORMALIZZAZIONE
logger.info("Loading preprocessed data...")
df_acc_clean = pd.read_parquet('../data/processed/acc_preprocessed_all.parquet')
df_acc_long  = df_acc_long
check(len(df_acc_clean) > 0, f"All signals loaded: {df_acc_clean['file'].nunique()} files")
check(len(df_acc_long) > 0,  f"Long signals loaded: {df_acc_long['file'].nunique()} files")

FIGURES_DIR = Path('../figures/03_single_signal')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
(FIGURES_DIR / 'scaling' / 'displacement' / 'event_window').mkdir(parents=True, exist_ok=True)
check(FIGURES_DIR.exists(), f"Figures directory ready: {FIGURES_DIR}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

def plot_empirical_pdfs_all_files(df_acc, bins=100, output_dir='../figures/acceleration_pdfs'):
    """
    Plot empirical PDFs for all files individually and aggregated.
    """
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    
    files = df_acc['file'].unique()
    n_files = len(files)
    
    # 1. INDIVIDUAL PDFs (grid plot)
    print(f"Plotting individual PDFs for {n_files} files...")
    
    n_cols = 6
    n_rows = int(np.ceil(n_files / n_cols))
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4*n_cols, 3*n_rows))
    axes = axes.flatten() if n_files > 1 else [axes]
    
    for i, file in enumerate(files):
        data = df_acc[df_acc['file'] == file]['acceleration'].values
        
        axes[i].hist(data, bins=bins, edgecolor='none', alpha=0.7, density=True)
        axes[i].axvline(0, color='red', linestyle='--', linewidth=1, alpha=0.5)
        axes[i].set_xlabel('Acceleration (cm/s²)', fontsize=8)
        axes[i].set_ylabel('Density', fontsize=8)
        
        # Stats in title
        mean_val = np.mean(data)
        std_val = np.std(data)
        max_val = np.max(np.abs(data))
        axes[i].set_title(f'{file.split(".")[1]}\nμ={mean_val:.3f}, σ={std_val:.3f}, max={max_val:.3f}', 
                         fontsize=8)
        axes[i].tick_params(labelsize=7)
    
    # Hide unused subplots
    for i in range(n_files, len(axes)):
        axes[i].set_visible(False)
    
    plt.suptitle('Empirical PDFs - Individual Files (Raw Acceleration)', fontsize=14)
    plt.tight_layout()
    plt.savefig(f'{output_dir}/pdfs_individual_grid.pdf', bbox_inches='tight')
    plt.close()
    print(f"✅ Saved: {output_dir}/pdfs_individual_grid.pdf")
    
    # 2. AGGREGATED PDF (all files together)
    print("Plotting aggregated PDF...")
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    all_data = df_acc['acceleration'].values
    
    # Linear scale
    axes[0].hist(all_data, bins=bins, edgecolor='none', alpha=0.7, density=True, 
                color='steelblue')
    axes[0].axvline(0, color='red', linestyle='--', linewidth=1.5, label='Zero')
    axes[0].set_xlabel('Acceleration (cm/s²)')
    axes[0].set_ylabel('Probability density')
    axes[0].set_title('Aggregated PDF - Linear scale')
    axes[0].legend()
    axes[0].grid(alpha=0.3)
    
    # Log scale
    axes[1].hist(all_data, bins=bins, edgecolor='none', alpha=0.7, density=True,
                color='steelblue')
    axes[1].axvline(0, color='red', linestyle='--', linewidth=1.5, label='Zero')
    axes[1].set_xlabel('Acceleration (cm/s²)')
    axes[1].set_ylabel('Probability density (log scale)')
    axes[1].set_title('Aggregated PDF - Log scale')
    axes[1].set_yscale('log')
    axes[1].legend()
    axes[1].grid(alpha=0.3)
    
    # Stats
    mean_all = np.mean(all_data)
    std_all = np.std(all_data)
    min_all = np.min(all_data)
    max_all = np.max(all_data)
    
    plt.suptitle(f'Empirical PDF - All Files Aggregated (Raw Acceleration)\n'
                f'μ={mean_all:.4f}, σ={std_all:.4f}, range=[{min_all:.3f}, {max_all:.3f}] cm/s²',
                fontsize=13)
    plt.tight_layout()
    plt.savefig(f'{output_dir}/pdf_aggregated.pdf', bbox_inches='tight')
    plt.close()
    print(f"✅ Saved: {output_dir}/pdf_aggregated.pdf")
    
    # 3. STATISTICS SUMMARY
    print("\n" + "="*70)
    print("STATISTICS SUMMARY")
    print("="*70)
    
    stats_per_file = []
    for file in files:
        data = df_acc[df_acc['file'] == file]['acceleration'].values
        stats_per_file.append({
            'file': file.split('.')[1],  # station name
            'mean': np.mean(data),
            'std': np.std(data),
            'min': np.min(data),
            'max': np.max(data),
            'abs_max': np.max(np.abs(data)),
            'n_samples': len(data)
        })
    
    df_stats = pd.DataFrame(stats_per_file)
    df_stats = df_stats.sort_values('abs_max', ascending=False)
    
    print("\nTop 10 files by |max acceleration|:")
    print(df_stats.head(10).to_string(index=False))
    
    print("\n\nOverall statistics:")
    print(f"  Mean:       {mean_all:.6f} cm/s²")
    print(f"  Std:        {std_all:.6f} cm/s²")
    print(f"  Min:        {min_all:.6f} cm/s²")
    print(f"  Max:        {max_all:.6f} cm/s²")
    print(f"  Range:      {max_all - min_all:.6f} cm/s²")
    
    print("\n  Percentage of values by range:")
    print(f"    |a| < 0.1:  {100*np.sum(np.abs(all_data) < 0.1)/len(all_data):.1f}%")
    print(f"    |a| < 0.5:  {100*np.sum(np.abs(all_data) < 0.5)/len(all_data):.1f}%")
    print(f"    |a| < 1.0:  {100*np.sum(np.abs(all_data) < 1.0)/len(all_data):.1f}%")
    print(f"    |a| >= 1.0: {100*np.sum(np.abs(all_data) >= 1.0)/len(all_data):.1f}%")
    
    print("="*70)
    
    return df_stats

# USO:
df_stats = plot_empirical_pdfs_all_files(df_acc_long, bins=100)